In [ ]:
library(Seurat)
# library(SeuratDisk)

library(reticulate)
library(anndata)

library(ggplot2)
library(ggpubr)
library(pheatmap)
library(dplyr)
library(tidyr)
library(RColorBrewer)
library(clustree)
library(repr)
options(repr.plot.width=10, repr.plot.height=8)

library(UpSetR)
library(grid)

library(PRROC)
library(Matrix)

getwd()

dataset_id <- "simulated_mm_RA"
genome_id <- "mm10"
samples <- c("all", "old", "young")
colorSamples <- c("old"="#A58065",
                "young"="#7FCFF2", 
                "all"="#92A8AC")
for (sample in samples) {
    dir.create(paste0("figures_", dataset_id, "_", sample))
}

colorTools <- c( # "MATES"="#F98A7B",
    "STARsolo_TE" = "#FFBC81",
    "STARsolo_TE_EM" = "#F3E088",
    "SoloTE_unique" = "#A4DD9B",
    "SoloTE_thr2" = "#6FC69D",
    "SoloTE_thr1" = "#4BB2BB",
    "SoloTE_thr0" = "#3989BF",
    "Stellarscope" = "#4F5D93",
    "Stellarscope_thr09" = "#2E3D72",
    "simulated" = "grey70"
)

thrMinCells <- 500 * 0.05

In [ ]:
#import workspace
save.image("workspaces/00_objectCreation_wStellarscopeThr.Rdata")

# Evaluation

## Upset plots of detected TEs and TPs

### Upsets of detected TEs

In [ ]:
c(names(objList), "simulated")


colorTools <- c( # "MATES"="#F98A7B",
    "STARsolo_TE" = "#FFBC81",
    "STARsolo_TE_EM" = "#F3E088",
    "SoloTE_unique" = "#A4DD9B",
    "SoloTE_thr2" = "#6FC69D",
    "SoloTE_thr1" = "#4BB2BB",
    "SoloTE_thr0" = "#3989BF",
    "Stellarscope" = "#4F5D93",
    "Stellarscope_thr09" = "#2E3D72",
    "simulated" = "grey70"
)

samples <- c("old", "young")

In [ ]:
options(repr.plot.width=13, repr.plot.height=8)

for (sample in samples){

    print(sample)

    TPlist <- list()
    detectedList <- list()

    tools_with_sample <- names(Filter(function(tool) sample %in% names(tool), objList))
    tools_with_sample <- names(colorTools)[names(colorTools) %in% tools_with_sample]
    
    print(tools_with_sample)
    
    for (tool in c(tools_with_sample, "simulated")){

        print(tool)
        if (tool == "simulated") {
            obj <- splatter_objs[[sample]]
        }else {
            obj <- objList[[tool]][[sample]]
        }
        print(sub("^(.*)_.*$", "\\1", obj@project.name))
        detectedList[[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- Features(obj)
        TPlist[[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- intersect(Features(splatter_objs[[sample]]), Features(obj))
    }
    print(names(detectedList))
    pdf(file=paste0("figures_",dataset_id,"_",sample,"/upset_detected_withStellarscopeThr09.pdf"), width = 13, height = 8)
    show(UpSetR::upset(fromList(detectedList), nintersects = 30,
            sets=c(tools_with_sample, "simulated"), keep.order = T, sets.bar.color=colorTools[c(tools_with_sample, "simulated")], 
            text.scale = c(2, 2, 2, 1.65, 2.5, 1.5), 
            order.by=c("freq"),
            point.size=3, mb.ratio = c(0.5, 0.5),
            set_size.show=F, set_size.scale_max= max(lengths(detectedList))+1000, #show.numbers = F,
            nsets=length(objList)+1,
            sets.x.label="N. detected loci",
            mainbar.y.label="Intersection size")
        
    )
    grid.text(paste0("Detected TEs - ", sample ),x = 0.65, y=0.95, gp=gpar(fontsize=18))
    dev.off()

    pdf(file=paste0("figures_",dataset_id,"_",sample,"/upset_detected_TP_withStellarscopeThr09.pdf"), width = 11, height = 8)
    show(UpSetR::upset(fromList(TPlist), nintersects = 20, 
            sets=c(tools_with_sample, "simulated"), keep.order = T, sets.bar.color=colorTools[c(tools_with_sample, "simulated")], 
            text.scale = c(2, 2, 2, 1.65, 2.2, 1.5), 
            order.by=c("freq"),
            point.size=3, mb.ratio = c(0.5, 0.5),
            set_size.show=F, set_size.scale_max= max(lengths(TPlist))+1000, #show.numbers = F, 
            nsets=length(c(objList, splatter_obj)),
            sets.x.label="N. correctly detected loci",
            mainbar.y.label="Intersection size")
    )
    grid.text(paste0("TP TEs - ", sample ),x = 0.65, y=0.95, gp=gpar(fontsize=18))
    
    dev.off()
}


## Precision / Recall


### Thr 0 counts

In [ ]:
precisionTool <- list()
recallTool <- list()

for (sample in samples){
    layer <- "counts"

    matSplatter <- GetAssayData(splatter_objs[[sample]], layer=layer)

    precisionTool[[sample]] <- list()
    recallTool[[sample]] <- list()

    for(tool in names(objList)){
        
        matTool <- GetAssayData(objList[[tool]][[sample]], layer=layer)
    
        precisionTool[[sample]][[tool]] <- NULL
        recallTool[[sample]][[tool]] <- NULL
        
        for(cell in Cells(splatter_objs[[sample]])){
            
            exprSplatter <- names(matSplatter[,cell])[matSplatter[,cell] > 0] 
            # TEs expressed in the simulated matrix
            
            exprTool <- names(matTool[,cell])[matTool[,cell] > 0] 
            # TEs detected by the tool

            TP <- intersect(exprSplatter, exprTool)
            nTP <- length(TP)
            
            FP <- setdiff(exprTool, exprSplatter)
            nFP <- length(FP)
            
            # add FP genes to the FP count
            if(tool %in% names(objGeneList)){
                FP_genes <- names(matTool_genes[,cell])[matTool_genes[,cell] > 0]
                nFP_genes <- length(FP_genes)
                nFP <- nFP + nFP_genes
            }
            
            FN <- setdiff(exprSplatter, exprTool)
            nFN <- length(FN)
            
            precision <- nTP / (nTP + nFP)
            precisionTool[[sample]][[tool]] <- c(precisionTool[[sample]][[tool]], precision)
            
            recall <- nTP / (nTP + nFN)
            recallTool[[sample]][[tool]] <- c(recallTool[[sample]][[tool]], recall)    
        }
    }
}

In [ ]:
options(repr.plot.width=5, repr.plot.height=3)

for (sample in samples){
    print(sample)

    precisionDf <- stack(precisionTool[[sample]])
    colnames(precisionDf) <- c("precision", "tool")

    recallDf <- stack(recallTool[[sample]])
    colnames(recallDf) <- c("recall", "tool")

    df <- merge(precisionDf, recallDf, by='tool')

    df$F1score <- (2 * df$precision * df$recall) / (df$precision + df$recall)

    df$tool <- factor(df$tool, levels=rev(names(colorTools)))

    # plot just the mean
    precisionDf <- stack(lapply(precisionTool[[sample]], mean))
    colnames(precisionDf) <- c("precision", "tool")

    recallDf <- stack(lapply(recallTool[[sample]], mean))
    colnames(recallDf) <- c("recall", "tool")

    Fscores <- sapply(names(precisionTool[[sample]]), function(tool){
                            (2 * precisionTool[[sample]][[tool]] * recallTool[[sample]][[tool]]) / 
                            (precisionTool[[sample]][[tool]] + recallTool[[sample]][[tool]])
    })
    meanFscores <- colMeans(Fscores)
    FscoreDf <- stack(meanFscores)
    colnames(FscoreDf) <- c("F1score", "tool")

    df <- merge(precisionDf, recallDf, by=c("tool"))
    df <- merge(df, FscoreDf, by=c("tool"))

    df$tool <- factor(df$tool, levels=rev(names(colorTools)))

}


In [ ]:
meanF1score_age_df <- NULL

for(age in c("old", "young")){

  # plot just the mean value
  precisionDf <- stack(lapply(precisionTool[[age]], mean))
  colnames(precisionDf) <- c("precision", "tool")

  recallDf <- stack(lapply(recallTool[[age]], mean))
  colnames(recallDf) <- c("recall", "tool")

  Fscores <- sapply(names(precisionTool[[age]]), function(tool){
                          (2 * precisionTool[[age]][[tool]] * recallTool[[age]][[tool]]) /
                          (precisionTool[[age]][[tool]] + recallTool[[age]][[tool]])
  })
  
  meanFscores <- colMeans(Fscores)
  FscoreDf <- stack(meanFscores)
  colnames(FscoreDf) <- c("F1score", "tool")

  df <- merge(precisionDf, recallDf, by=c("tool"))
  df <- merge(df, FscoreDf, by=c("tool"))

  df$tool <- factor(df$tool, levels=rev(names(colorTools)))
  
  # create df for facet plot
  meanF1score_age_df <- rbind(meanF1score_age_df, cbind(df, age)) 

}


In [ ]:
options(repr.plot.width=9, repr.plot.height=4)

ggplot(meanF1score_age_df, aes(x=F1score, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 2) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean F1 score")) +
    guides(fill="none", color="none") +
  xlim(c(0,1)) +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))
ggsave(paste0("figures_",dataset_id,"/meanF1_tool_byAge_horiz_withStellarscopeThr09.pdf"), device="pdf", height=3.2, width=7)

In [ ]:
options(repr.plot.width=5, repr.plot.height=6)

ggplot(meanF1score_age_df, aes(x=F1score, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean F1 score")) +
    guides(fill="none", color="none") +
  xlim(c(0,1)) +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))
ggsave(paste0("figures_",dataset_id,"/meanF1_tool_byAge_vert_withStellarscopeThr09.pdf"), device="pdf", height=6, width=5)

In [ ]:
options(repr.plot.width=8, repr.plot.height=6)

precision_plot <- ggplot(meanF1score_age_df, aes(x=precision, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean precision")) +
    guides(fill="none", color="none") +
    xlim(c(0,1)) +
    ylab("") +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))

recall_plot <- ggplot(meanF1score_age_df, aes(x=recall, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean recall")) +
    guides(fill="none", color="none") +
    xlim(c(0,1)) +
    ylab("") +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_blank(), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))

f1_plot <- ggplot(meanF1score_age_df, aes(x=F1score, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean F1 score")) +
    guides(fill="none", color="none") +
    xlim(c(0,1)) +
    ylab("") +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_blank(), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))

p <- precision_plot + recall_plot
p 

ggsave(paste0("figures_simulated_mm_RA/meanPrecisionRecall_tool_byAge_withStellarscopeThr09.pdf"), 
      device="pdf", height=6, width=8)


In [ ]:
options(repr.plot.width=11, repr.plot.height=6)

p + f1_plot 

ggsave(paste0("figures_simulated_mm_RA/meanPrecisionRecall_tool_byAge_withStellarscopeThr09.pdf"), 
      device="pdf", height=6, width=11)